# Knowledge distillation : un code complet, en classes

**But** : entraîner un petit **élève** (MLP) à imiter un **professeur** (CNN) sur MNIST,
et vérifier qu'il fait mieux que le même élève entraîné sur les seuls labels.

Organisation, dans l'esprit du TP4 :
1. `DistillationLoss` : la perte $\alpha\,\text{CE} + (1-\alpha)\,T^2\,\text{KL}$ ;
2. `Classifier` : la partie commune (optimiseur, `training_step`, `validation_step`) ;
3. `TeacherCNN`, `StudentMLP` : les deux architectures ;
4. `DistilledStudent` : l'élève qui apprend du professeur figé ;
5. `Trainer` : boucle d'entraînement, validation à chaque epoch, meilleure epoch gardée ;
6. le **test**, une seule fois, à la fin.

Dans Colab : Exécution, Modifier le type d'exécution, GPU (facultatif, le CPU suffit).

In [ ]:
import copy, os, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
QUICK = False   # True pour un test très rapide (petits sous-ensembles, 1 epoch)
print("device :", device)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 1. Données : train, validation, test

MNIST : 60 000 images d'entraînement, que l'on coupe en 50 000 (train) et 10 000 (validation),
et 10 000 images de test, regardées **une seule fois** à la fin.

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST("data", train=True, download=True, transform=tf)
test_data = datasets.MNIST("data", train=False, download=True, transform=tf)

set_seed(0)
train_data, val_data = random_split(full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(0))
if QUICK:
    train_data = torch.utils.data.Subset(train_data, range(2_000))
    val_data = torch.utils.data.Subset(val_data, range(1_000))
    test_data = torch.utils.data.Subset(test_data, range(1_000))

train_dl = DataLoader(train_data, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(0))
val_dl = DataLoader(val_data, batch_size=512)
test_dl = DataLoader(test_data, batch_size=512)
print(len(train_data), len(val_data), len(test_data))

## 2. La perte de distillation

$$\mathcal{L} = \alpha\,\text{CE}(y, p_S) + (1-\alpha)\,T^2\,\text{KL}\big(p_T^{(T)} \,\|\, p_S^{(T)}\big)$$

- `F.kl_div` attend les **log-probabilités** de l'élève et les **probabilités** du professeur ;
- le facteur $T^2$ compense les gradients du terme adouci, qui sont d'ordre $1/T^2$.

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.3):
        super().__init__()
        self.T, self.alpha = T, alpha

    def forward(self, s_logits, t_logits, y):
        hard = F.cross_entropy(s_logits, y)                              # vrais labels, T = 1
        soft = F.kl_div(F.log_softmax(s_logits / self.T, dim=1),          # élève : LOG-probas
                        F.softmax(t_logits / self.T, dim=1),              # professeur : probas
                        reduction="batchmean") * self.T ** 2
        return self.alpha * hard + (1 - self.alpha) * soft

## 3. Les modèles

`Classifier` regroupe ce qui est commun (même design que le TP4) : `configure_optimizers`,
`training_step` (entropie croisée sur les labels) et `validation_step` (perte sommée, prédictions).

In [ ]:
class Classifier(nn.Module):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.lr = lr

    def configure_optimizers(self):
        return torch.optim.Adam([p for p in self.parameters() if p.requires_grad], lr=self.lr)

    def training_step(self, batch):
        X, y = batch
        return F.cross_entropy(self(X), y)

    def validation_step(self, batch):
        X, y = batch
        with torch.no_grad():
            logits = self(X)
            loss = F.cross_entropy(logits, y, reduction="sum").item()
        return loss, logits.argmax(1), y


class TeacherCNN(Classifier):
    """Professeur : un CNN, assez gros pour être précis."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 32 x 14 x 14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64 x 7 x 7
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.net(x)


class StudentMLP(Classifier):
    """Élève : un tout petit MLP (une couche cachée de 32 neurones)."""
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, hidden), nn.ReLU(), nn.Linear(hidden, 10))

    def forward(self, x):
        return self.net(x)


class DistilledStudent(Classifier):
    """Élève qui apprend du professeur figé : seule la perte d'entraînement change."""
    def __init__(self, student, teacher, T=4.0, alpha=0.3):
        super().__init__(lr=student.lr)
        self.student = student
        self.teacher = teacher.eval()                  # mode eval : pas de dropout chez le professeur
        for p in self.teacher.parameters():
            p.requires_grad = False                    # professeur figé
        self.loss_fn = DistillationLoss(T, alpha)

    def forward(self, x):
        return self.student(x)                         # à l'inférence, seul l'élève sert (T = 1)

    def train(self, mode=True):
        super().train(mode)
        self.teacher.eval()                            # le professeur reste en eval même pendant model.train()
        return self

    def training_step(self, batch):
        X, y = batch
        with torch.no_grad():
            t_logits = self.teacher(X)
        return self.loss_fn(self.student(X), t_logits, y)


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("professeur :", n_params(TeacherCNN()), "paramètres ; élève :", n_params(StudentMLP()), "paramètres")

## 4. Entraînement et validation

Le `Trainer` du TP4, avec un historique par epoch et les poids de la **meilleure epoch en validation**.

In [ ]:
def evaluate(dataloader, model):
    model.eval()
    size = len(dataloader.dataset)
    loss, correct = 0.0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        batch_loss, pred, y_true = model.validation_step((X, y))
        loss += batch_loss
        correct += (pred == y_true).sum().item()
    return loss / size, correct / size


class Trainer:
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs

    def fit(self, model, train_dl, val_dl, name=""):
        model.to(device)
        optimizer = model.configure_optimizers()
        history = {"train_loss": [], "val_loss": [], "val_acc": []}
        best_acc, best_state = -1.0, None
        for epoch in range(self.max_epochs):
            model.train()
            running, n = 0.0, 0
            for X, y in train_dl:
                X, y = X.to(device), y.to(device)
                loss = model.training_step((X, y))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running += loss.item(); n += 1
            val_loss, val_acc = evaluate(val_dl, model)
            history["train_loss"].append(running / n)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            print(f"{name:12s} epoch {epoch + 1} : train loss {running / n:.3f} | val loss {val_loss:.3f} | val acc {val_acc:.4f}")
            if val_acc > best_acc:
                best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        return history

## 5. Les trois entraînements

1. le **professeur** sur les labels ;
2. l'**élève seul** sur les labels (référence) ;
3. l'**élève distillé**, même architecture, même seed, mêmes epochs : seule la perte change.

In [ ]:
E_TEACHER = 1 if QUICK else 3
E_STUDENT = 1 if QUICK else 10

set_seed(0)
teacher = TeacherCNN()
Trainer(E_TEACHER).fit(teacher, train_dl, val_dl, "professeur")

set_seed(1)
baseline = StudentMLP()
Trainer(E_STUDENT).fit(baseline, train_dl, val_dl, "élève seul")

set_seed(1)                                            # même initialisation que l'élève seul
distilled = DistilledStudent(StudentMLP(), teacher, T=4.0, alpha=0.3)
Trainer(E_STUDENT).fit(distilled, train_dl, val_dl, "élève distillé")

## 6. Test : une seule fois, à la fin

In [ ]:
for name, model in [("professeur", teacher), ("élève seul", baseline), ("élève distillé", distilled)]:
    _, acc = evaluate(test_dl, model)
    print(f"TEST {name:14s} : accuracy {100 * acc:.2f} %")

## 7. Et avec peu de données ?

Avec 50 000 images, l'élève seul a déjà presque tout ce qu'il lui faut : la distillation apporte peu.
Son intérêt apparaît quand l'élève a **peu d'exemples labellisés** : le professeur, entraîné sur
toutes les données, lui transmet ce que 5 000 images ne suffisent pas à apprendre.

In [ ]:
small = torch.utils.data.Subset(train_data, range(500 if QUICK else 5_000))
small_dl = DataLoader(small, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(0))
E_SMALL = 1 if QUICK else 30

set_seed(1)
baseline_small = StudentMLP()
Trainer(E_SMALL).fit(baseline_small, small_dl, val_dl, "seul (5k)")

set_seed(1)
distilled_small = DistilledStudent(StudentMLP(), teacher, T=4.0, alpha=0.3)
Trainer(E_SMALL).fit(distilled_small, small_dl, val_dl, "distillé (5k)")

for name, model in [("élève seul, 5 000 images", baseline_small), ("élève distillé, 5 000 images", distilled_small)]:
    _, acc = evaluate(test_dl, model)
    print(f"TEST {name:30s} : accuracy {100 * acc:.2f} %")

## À tester ensuite

- Changer la température `T` (1, 2, 4, 8) et le poids `alpha` (0, 0,3, 0,7, 1) : avec `alpha = 1`, on retrouve l'élève seul.
- Répéter avec 3 seeds et comparer moyenne ± écart-type : un écart plus petit que la variabilité entre seeds ne prouve rien.
- Entraîner l'élève distillé plus longtemps sur toutes les données : il progressait encore à la dernière epoch.